# Presentation Videos v2 — Diffusion Field (Case 4)

**Case 4a:** `dicty_spring_force_rk4_diffusion_field_v1` — GNN + SIREN (vector)
**Case 4b:** `dicty_spring_force_rk4_diffusion_field_siren_grad_v2` — GNN + SIREN grad (scalar)
**Case 4c:** `dicty_spring_force_rk4_diffusion_field_siren_grad_v5` — GNN + SIREN grad (scalar)

Each case: simulation 3D, reconstruction 3D, force (MLP1), field learning, v_pred scatter, loss curve.

In [ ]:
import sys, os, re

PROJ = '/groups/jingyiliu/home/liuj4/cell-gnn'
sys.path.insert(0, PROJ)
os.chdir(PROJ)

import torch
import numpy as np
import zarr
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from pathlib import Path
import subprocess, shutil

from cell_gnn.config import CellGNNConfig
from cell_gnn.utils import to_numpy
from cell_gnn.models.Siren_Network import Siren
from cell_gnn.models.MLP import MLP
from cell_gnn.plot import build_edge_features, _batched_mlp_eval

PROJ = Path(PROJ)
VID_DIR = PROJ / 'presentation' / 'videos'
VID_DIR.mkdir(parents=True, exist_ok=True)

def load_config(n):
    return CellGNNConfig.from_yaml(str(PROJ / 'config' / 'misc' / f'{n}.yaml'))
def load_pos(d):
    return np.array(zarr.open(str(PROJ/'graphs_data'/'misc'/d/'x_list_0'/'pos.zarr'),'r')[:])
def load_field(d):
    return np.array(zarr.open(str(PROJ/'graphs_data'/'misc'/d/'x_list_0'/'field.zarr'),'r')[:])
def load_vel(d):
    return np.array(zarr.open(str(PROJ/'graphs_data'/'misc'/d/'x_list_0'/'vel.zarr'),'r')[:])

def load_checkpoint(config_name):
    md = PROJ / 'log' / 'misc' / config_name / 'models'
    ckpts = sorted(md.glob('best_model_with_0_graphs_0_*.pt'),
                   key=lambda p: int(p.stem.split('_')[-1]))
    if not ckpts:
        ckpts = list(md.glob('best_model_with_0_graphs_0.pt'))
    raw = torch.load(ckpts[-1], map_location='cpu', weights_only=False)
    state = raw.get('model_state_dict', raw) if isinstance(raw, dict) else raw
    return {k.replace('_orig_mod.', ''): v for k, v in state.items()}

def all_checkpoints(config_name):
    md = PROJ / 'log' / 'misc' / config_name / 'models'
    pat = re.compile(r'best_model_with_0_graphs_(\d+)_(\d+)\.pt$')
    results = []
    for p in md.glob('best_model_with_0_graphs_*_*.pt'):
        m = pat.search(str(p))
        if m:
            results.append((int(m.group(1)), int(m.group(2)), p))
    results.sort(key=lambda x: (x[0], x[1]))
    return [(f'Ep {ep} / {it:,}', p) for ep, it, p in results]

def get_particle_radius(config_name):
    cfg = load_config(config_name)
    r0 = cfg.simulation.cell_params[0][1]
    return r0 * 0.3

def data_to_scatter_size(radius, fig_width_inches, dpi, data_range=1.0, axis_fraction=0.65):
    fig_width_pts = fig_width_inches * 72
    axis_width_pts = fig_width_pts * axis_fraction
    pts_per_data = axis_width_pts / data_range
    radius_pts = radius * pts_per_data
    return np.pi * radius_pts ** 2

def frames_to_mp4(frame_dir, output_path, fps=30):
    subprocess.run(['ffmpeg','-y','-loglevel','error','-framerate',str(fps),
        '-i',f'{frame_dir}/frame_%06d.png',
        '-vf','scale=trunc(iw/2)*2:trunc(ih/2)*2',
        '-c:v','libx264','-crf','23','-pix_fmt','yuv420p',
        str(output_path)], check=True)
    print(f'  -> {output_path.name}')

def clean_3d_ax(ax):
    ax.grid(False)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False
    ax.xaxis.pane.set_edgecolor('k')
    ax.yaxis.pane.set_edgecolor('k')
    ax.zaxis.pane.set_edgecolor('k')
    ax.set_xlim(0, 1); ax.set_ylim(0, 1); ax.set_zlim(0, 1)
    ax.set_box_aspect([1, 1, 1])
    ax.set_xlabel('x'); ax.set_ylabel('y'); ax.set_zlabel('z')

def data_exists(d):
    return (PROJ/'graphs_data'/'misc'/d/'x_list_0'/'pos.zarr').exists()
def model_exists(c):
    return len(all_checkpoints(c)) > 0
def has_siren(c):
    cfg = load_config(c)
    return 'siren' in cfg.graph_model.cell_model_name

print('Setup done.')

---
## Video functions

In [ ]:
def make_simulation_3d(dataset_name, config_name, title, output_name,
                       frame_step=50, fps=30, max_frames=160):
    """3D rotating scatter colored by field value."""
    pos = load_pos(dataset_name)
    field = load_field(dataset_name)
    T = pos.shape[0]
    idxs = np.arange(0, min(T, max_frames * frame_step), frame_step)
    
    particle_r = get_particle_radius(config_name)
    fig_w = 7; dpi = 120
    s = data_to_scatter_size(particle_r, fig_w, dpi)
    
    f_all = field[idxs, :, 0]
    nz = f_all[f_all != 0]
    use_field = len(nz) > 0
    if use_field:
        vmin, vmax = np.percentile(nz, [2, 98])
        if vmin == vmax: vmin, vmax = -1, 1
    
    tmp = VID_DIR / f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    fig = plt.figure(figsize=(fig_w, fig_w))
    ax = fig.add_subplot(111, projection='3d')
    
    for i, fi in enumerate(idxs):
        ax.clear()
        p = pos[fi]
        if use_field:
            f = field[fi, :, 0]
            ax.scatter(p[:,0], p[:,1], p[:,2], c=f, s=s, alpha=0.7,
                       cmap='coolwarm', vmin=vmin, vmax=vmax,
                       edgecolors='none', depthshade=True)
        else:
            ax.scatter(p[:,0], p[:,1], p[:,2], s=s, alpha=0.7,
                       c='#1f77b4', edgecolors='none', depthshade=True)
        clean_3d_ax(ax)
        ax.set_title(f'{title}\n$t = {fi*0.002:.2f}$', fontsize=12)
        ax.view_init(elev=25, azim=30 + i * 0.5)
        fig.savefig(tmp / f'frame_{i:06d}.png', dpi=dpi, bbox_inches='tight')
        if i % 50 == 0: print(f'  {i}/{len(idxs)}')
    plt.close(fig)
    frames_to_mp4(tmp, VID_DIR / f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

In [ ]:
def make_reconstruction_3d(dataset_name, config_name, title, output_name,
                           frame_step=50, fps=20, max_frames=160, n_rollout=10):
    """Side-by-side 3D: GT (blue) vs model rollout (red)."""
    pos_all = load_pos(dataset_name)
    T, N, _ = pos_all.shape
    dt = 0.002
    
    cfg = load_config(config_name)
    dim = cfg.simulation.dimension
    emb_dim = cfg.graph_model.embedding_dim
    max_radius = cfg.simulation.max_radius
    isz = dim + 1 + emb_dim
    
    particle_r = get_particle_radius(config_name)
    fig_w = 14; dpi = 120
    s = data_to_scatter_size(particle_r, fig_w / 2, dpi)
    
    state = load_checkpoint(config_name)
    lin_edge = MLP(input_size=isz, output_size=cfg.graph_model.output_size,
                   nlayers=cfg.graph_model.n_layers, hidden_size=cfg.graph_model.hidden_dim,
                   device='cpu')
    es = {k.replace('lin_edge.',''):v for k,v in state.items() if k.startswith('lin_edge.')}
    lin_edge.load_state_dict(es); lin_edge.eval()
    med_emb = state['a'][0].median(dim=0).values if 'a' in state else torch.zeros(emb_dim)
    
    def predict_vel(pos_t):
        N = pos_t.shape[0]
        dist = torch.cdist(pos_t, pos_t)
        pred = torch.zeros(N, 3)
        for ci in range(N):
            nbrs = (dist[ci] < max_radius) & (dist[ci] > 0)
            if not nbrs.any(): continue
            dp = (pos_t[nbrs] - pos_t[ci]) / max_radius
            r = dist[ci, nbrs].unsqueeze(1) / max_radius
            emb = med_emb.unsqueeze(0).expand(dp.shape[0], -1)
            inp = torch.cat([dp, r, emb], dim=-1)
            pred[ci] = lin_edge(inp).sum(dim=0)
        return pred
    
    idxs = np.arange(0, min(T - n_rollout, max_frames * frame_step), frame_step)
    tmp = VID_DIR / f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    fig = plt.figure(figsize=(fig_w, 6))
    ax1 = fig.add_subplot(121, projection='3d')
    ax2 = fig.add_subplot(122, projection='3d')
    
    for i, fi in enumerate(idxs):
        with torch.no_grad():
            cur = torch.tensor(pos_all[fi], dtype=torch.float32)
            for _ in range(n_rollout):
                vel = predict_vel(cur)
                cur = (cur + vel * dt) % 1.0
        pred_pos = cur.numpy()
        gt_pos = pos_all[fi + n_rollout]
        
        ax1.clear(); ax2.clear()
        ax1.scatter(gt_pos[:,0], gt_pos[:,1], gt_pos[:,2],
                   s=s, alpha=0.7, c='#1f77b4', edgecolors='none', depthshade=True)
        clean_3d_ax(ax1); ax1.set_title('Ground Truth', fontsize=12)
        ax1.view_init(elev=25, azim=30 + i*0.5)
        
        ax2.scatter(pred_pos[:,0], pred_pos[:,1], pred_pos[:,2],
                   s=s, alpha=0.7, c='#d62728', edgecolors='none', depthshade=True)
        clean_3d_ax(ax2); ax2.set_title(f'Reconstructed ({n_rollout}-step)', fontsize=12)
        ax2.view_init(elev=25, azim=30 + i*0.5)
        
        fig.suptitle(f'{title}   $t = {fi*dt:.2f}$', fontsize=13)
        fig.tight_layout()
        fig.savefig(tmp / f'frame_{i:06d}.png', dpi=dpi, bbox_inches='tight')
        if i % 10 == 0: print(f'  {i}/{len(idxs)}')
    
    plt.close(fig)
    frames_to_mp4(tmp, VID_DIR / f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

In [ ]:
def stitch_pngs_to_video(png_dir, pattern, output_name, fps=8, target_frames=30):
    """Stitch existing PNGs (sorted by epoch_iteration) into a video."""
    pat = re.compile(pattern)
    pngs = []
    for p in Path(png_dir).glob('*.png'):
        m = pat.search(p.name)
        if m:
            pngs.append((int(m.group(1)), int(m.group(2)), p))
    pngs.sort(key=lambda x: (x[0], x[1]))
    
    if not pngs:
        print(f'  No matching PNGs in {png_dir}'); return
    print(f'  Found {len(pngs)} plots')
    
    pngs = pngs[::max(1, len(pngs) // target_frames)]
    print(f'  Using {len(pngs)} frames')
    
    tmp = VID_DIR / f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    for i, (ep, it, src) in enumerate(pngs):
        shutil.copy2(src, tmp / f'frame_{i:06d}.png')
    
    frames_to_mp4(tmp, VID_DIR / f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

In [ ]:
def make_loss_curve(config_name, output_name):
    """Save training loss curve as PNG."""
    log_dir = PROJ / 'log' / 'misc' / config_name
    loss_path = log_dir / 'loss.pt'
    if not loss_path.exists():
        print(f'  No loss.pt'); return
    
    loss_data = torch.load(loss_path, map_location='cpu', weights_only=False)
    if isinstance(loss_data, list):
        loss_vals = np.array(loss_data)
    elif isinstance(loss_data, dict):
        loss_vals = np.array(list(loss_data.values())[0])
    else:
        loss_vals = to_numpy(torch.tensor(loss_data))
    
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))
    ax.plot(np.arange(len(loss_vals)), loss_vals, 'b-o', lw=2, markersize=4)
    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'Training Loss', fontsize=11)
    ax.grid(True, alpha=0.3)
    if len(loss_vals) > 2 and loss_vals.max() / max(loss_vals.min(), 1e-10) > 5:
        ax.set_yscale('log')
    fig.tight_layout()
    out_path = VID_DIR / f'{output_name}.png'
    fig.savefig(out_path, dpi=150, bbox_inches='tight')
    plt.close(fig)
    print(f'  -> {out_path.name}')

In [ ]:
def run_case4(config_name, label, prefix):
    """Run all videos for a diffusion field case:
    simulation, reconstruction, force (MLP1), field learning, v_pred, loss."""
    dataset_name = config_name  # dataset name = config name
    log_dir = PROJ / 'log' / 'misc' / config_name
    tmp_dir = log_dir / 'tmp_training'
    
    print(f'\n{"="*60}')
    print(f'{label}')
    print(f'  config: {config_name}')
    print(f'{"="*60}')
    
    if not data_exists(dataset_name):
        print('  ** Data not generated yet — skipping'); return
    if not model_exists(config_name):
        print('  ** Model not trained yet — skipping'); return
    
    # 1. Simulation 3D
    print(f'\n--- Simulation ---')
    make_simulation_3d(dataset_name, config_name, label, f'{prefix}_sim')
    
    # 2. Reconstruction 3D
    print(f'\n--- Reconstruction ---')
    make_reconstruction_3d(dataset_name, config_name, label, f'{prefix}_recon', n_rollout=5)
    
    # 3. Force learning (MLP1) — stitch existing PNGs
    print(f'\n--- Force learning (MLP1) ---')
    mlp1_dir = tmp_dir / 'function' / 'MLP1'
    if mlp1_dir.exists():
        stitch_pngs_to_video(mlp1_dir, r'function_(\d+)_(\d+)\.png$', f'{prefix}_force')
    else:
        print('  No MLP1 plots')
    
    # 4. Field learning — stitch existing field_fixed PNGs
    print(f'\n--- Field learning ---')
    field_dir = tmp_dir / 'field_fixed'
    if field_dir.exists():
        stitch_pngs_to_video(field_dir, r'(\d+)_(\d+)\.png$', f'{prefix}_field')
    else:
        print('  No field_fixed plots')
    
    # 5. v_pred scatter — stitch existing prediction PNGs
    print(f'\n--- v_pred scatter ---')
    pred_dir = tmp_dir / 'prediction'
    if pred_dir.exists():
        stitch_pngs_to_video(pred_dir, r'(\d+)_(\d+)\.png$', f'{prefix}_vpred')
    else:
        print('  No prediction plots')
    
    # 6. Loss curve
    print(f'\n--- Loss curve ---')
    make_loss_curve(config_name, f'{prefix}_loss')
    
    print(f'\nDone: {label}\n')

---
## Case 4a: `dicty_spring_force_rk4_diffusion_field_v1`

In [ ]:
run_case4(
    'dicty_spring_force_rk4_diffusion_field_v1',
    'Case 4a: Diffusion Field — GNN + SIREN (vector)',
    'case4a',
)


Case 4a: Diffusion Field — GNN + SIREN (vector)
  config: dicty_spring_force_rk4_diffusion_field_v1

--- Simulation ---
  0/160
  50/160
  100/160
  150/160
  -> case4a_sim.mp4

--- Reconstruction ---
  0/160
  10/160
  20/160
  30/160
  40/160
  50/160
  60/160
  70/160
  80/160
  90/160
  100/160
  110/160
  120/160
  130/160
  140/160
  150/160
  -> case4a_recon.mp4

--- Force learning (MLP1) ---
  Found 200 plots
  Using 34 frames
  -> case4a_force.mp4

--- Field learning ---
  Found 200 plots
  Using 34 frames
  -> case4a_field.mp4

--- v_pred scatter ---
  Found 200 plots
  Using 34 frames
  -> case4a_vpred.mp4

--- Loss curve ---
  -> case4a_loss.png

Done: Case 4a: Diffusion Field — GNN + SIREN (vector)



---
## Ground-truth diffusion field videos (3-panel, test_diffusion_field_generator style)

One video per case using each config's actual saved positions + field values,
with the PDE re-simulated to get the grid z=0.5 slice at each snapshot.

- Left: 3D particles colored by local field value (YlOrRd)
- Center: top-down particles colored by field
- Right: top-down field slice (Blues) + particles

In [ ]:
# Weber-Fechner saturation (v2)
make_gt_field_3panel_video(
    'dicty_spring_force_rk4_diffusion_field_siren_grad_v2',
    'case4_gt_field_saturation',
    'GT Diffusion Field — Weber-Fechner saturation (v2)',
)

In [ ]:
# Linear chemotaxis (v1 = v5 physics)
make_gt_field_3panel_video(
    'dicty_spring_force_rk4_diffusion_field_v1',
    'case4_gt_field_linear',
    'GT Diffusion Field — linear chemotaxis (v1/v5)',
)

In [ ]:
from cell_gnn.cell_state import CellState
from cell_gnn.utils import edges_radius_blockwise, choose_boundary_values
from cell_gnn.generators.particle_spring_force_diffusion_field import ParticleSpringForceDiffusionField
from tqdm import trange
import yaml

def _load_cfg_dict(name):
    with open(PROJ / 'config' / 'misc' / f'{name}.yaml') as f:
        return yaml.safe_load(f)

# Matched to other v2 videos: frame_step=50, max_frames=160, fps=30
# => 160 frames covering t=0..16, same pace as sim/recon videos
def make_gt_field_3panel_video(config_name, output_name, label,
                                frame_step=50, max_frames=160, fps=30, device='cpu'):
    """Generate 3-panel GT diffusion field video (test_diffusion_field_generator style).
    
    Timing matches make_simulation_3d / make_reconstruction_3d:
    - frame_step=50  (same dt interval)
    - max_frames=160 (same total frame count)
    - fps=30
    
    Uses saved pos.zarr + field.zarr, re-simulates PDE to get grid z=0.5 slice.
    """
    print(f'\n=== {label} ===')
    print(f'  config: {config_name}')
    cfg = _load_cfg_dict(config_name)
    sim = cfg['simulation']; fp = sim['field_params']
    
    n_cells = sim['n_cells']; dim = sim['dimension']
    n_frames_cfg = sim['n_frames']; dt = sim['delta_t']
    cell_params = torch.tensor(sim['cell_params'][0], device=device)
    res = fp['grid_resolution']
    
    pos_all = load_pos(config_name)
    field_all = load_field(config_name)
    T_saved = pos_all.shape[0]
    print(f'  {T_saved} saved frames, {n_cells} cells')
    
    sim_fp = dict(
        diffusion_coeff=fp['diffusion_coeff'], lambda_decay=fp['lambda_decay'],
        grid_resolution=res, source_strength=fp.get('source_strength', 0.0),
        source_fraction=fp.get('source_fraction', 1.0),
        pulse_period=fp.get('pulse_period', 0), pulse_duty=fp.get('pulse_duty', 1.0),
        center_0=torch.tensor(fp['center_0'], device=device),
        amplitude=fp['amplitude'], sigma=fp['sigma'], mu_chem=fp['mu_chem'],
        delta_t=dt, periodic=True,
    )
    sat = fp.get('chem_saturation_scale')
    if sat is not None:
        sim_fp['chem_saturation_scale'] = sat
    
    bc_pos, bc_dpos = choose_boundary_values('periodic')
    model = ParticleSpringForceDiffusionField(
        aggr_type='add', p=cell_params, bc_dpos=bc_dpos,
        dimension=dim, noise_model_level=0.0, field_params=sim_fp,
    )
    model._init_grid(device)
    
    # Indices at same spacing as make_simulation_3d
    n_steps = min(n_frames_cfg, T_saved, max_frames * frame_step)
    snap_idxs = list(range(0, n_steps, frame_step))
    
    # Replay PDE 1 step at a time (needed because the solver is stateful)
    pos_snaps, field_snaps, grid_snaps, t_snaps = [], [], [], []
    snap_set = set(snap_idxs)
    
    for it in trange(n_steps, ncols=100, desc='  PDE replay'):
        pos_t = torch.tensor(pos_all[it], dtype=torch.float32, device=device)
        model._advance_field(pos_t, n_steps=1)
        model._last_step = it
        if it in snap_set:
            pos_snaps.append(pos_all[it])
            field_snaps.append(field_all[it, :, 0])
            grid = model._field_grid.detach().cpu().numpy()
            grid_snaps.append(grid[res // 2, :, :].copy())
            t_snaps.append(it * dt)
    
    pos_snaps = np.array(pos_snaps)
    field_snaps = np.array(field_snaps)
    grid_snaps = np.array(grid_snaps)
    t_snaps = np.array(t_snaps)
    n_snap = len(t_snaps)
    print(f'  {n_snap} snapshots, field range [{grid_snaps.min():.4f}, {grid_snaps.max():.4f}]')
    
    # Render 3-panel
    vmax_field = grid_snaps.max()
    vmax_part = max(field_snaps.max(), 1e-6)
    
    tmp = VID_DIR / f'_tmp_{output_name}'; tmp.mkdir(exist_ok=True)
    fig = plt.figure(figsize=(18, 6))
    ax1 = fig.add_subplot(131, projection='3d')
    ax2 = fig.add_subplot(132)
    ax3 = fig.add_subplot(133)
    
    for i in trange(n_snap, ncols=100, desc='  rendering'):
        p = pos_snaps[i]; t = t_snaps[i]
        f = field_snaps[i]; g = grid_snaps[i]
        
        ax1.cla()
        ax1.scatter(p[:,0], p[:,1], p[:,2], s=8, c=f, cmap='YlOrRd',
                   vmin=0, vmax=vmax_part, alpha=0.7, edgecolors='none', depthshade=True)
        clean_3d_ax(ax1)
        ax1.set_title(f'3D  t={t:.2f}')
        ax1.view_init(elev=25, azim=30 + i*0.5)
        
        ax2.cla()
        ax2.scatter(p[:,0], p[:,1], s=4, c=f, cmap='YlOrRd',
                   vmin=0, vmax=vmax_part, alpha=0.7, edgecolors='none')
        ax2.set_xlim(0,1); ax2.set_ylim(0,1); ax2.set_aspect('equal')
        ax2.set_xlabel('x'); ax2.set_ylabel('y')
        ax2.set_title('Top-down (color = local c)')
        
        ax3.cla()
        ax3.imshow(g, extent=[0,1,0,1], origin='lower',
                  cmap='Blues', vmin=0, vmax=vmax_field, alpha=0.8)
        ax3.scatter(p[:,0], p[:,1], s=2, c='red', alpha=0.4)
        ax3.set_xlim(0,1); ax3.set_ylim(0,1); ax3.set_aspect('equal')
        ax3.set_xlabel('x'); ax3.set_ylabel('y')
        ax3.set_title('Field (z=0.5) + particles')
        
        fig.suptitle(f'{label}   t={t:.2f}', fontsize=14)
        fig.tight_layout()
        fig.savefig(tmp / f'frame_{i:06d}.png', dpi=100, bbox_inches='tight')
    
    plt.close(fig)
    frames_to_mp4(tmp, VID_DIR / f'{output_name}.mp4', fps)
    shutil.rmtree(tmp)

In [ ]:
make_gt_field_3panel_video(
    'dicty_spring_force_rk4_diffusion_field_v1',
    'case4a_gt_field',
    'Case 4a: GT Diffusion Field (v1, linear chemotaxis)',
)

In [ ]:
run_case4(
    'dicty_spring_force_rk4_diffusion_field_siren_grad_v5',
    'Case 4c: Diffusion Field — GNN + SIREN grad (v5)',
    'case4c',
)


Case 4c: Diffusion Field — GNN + SIREN grad (v5)
  config: dicty_spring_force_rk4_diffusion_field_siren_grad_v5

--- Simulation ---
  0/160
  50/160
  100/160
  150/160
  -> case4c_sim.mp4

--- Reconstruction ---
  0/160
  10/160
  20/160
  30/160
  40/160
  50/160
  60/160
  70/160
  80/160
  90/160
  100/160
  110/160
  120/160
  130/160
  140/160
  150/160
  -> case4c_recon.mp4

--- Force learning (MLP1) ---
  Found 200 plots
  Using 34 frames
  -> case4c_force.mp4

--- Field learning ---
  Found 200 plots
  Using 34 frames
  -> case4c_field.mp4

--- v_pred scatter ---
  Found 200 plots
  Using 34 frames
  -> case4c_vpred.mp4

--- Loss curve ---
  -> case4c_loss.png

Done: Case 4c: Diffusion Field — GNN + SIREN grad (v5)

